# 01 — Data Inspection

This notebook documents our first look at the raw CICIDS2017 dataset before any processing.

**Goals:**
- Confirm the CSV files are present and readable
- Understand the schema (feature names, types, label column)
- Spot data quality issues (whitespace in column names, inf values, nulls, encoding artefacts)
- Document class distribution across all 8 source files

**Output of this notebook:** a clear picture of what we're working with before `ingest.py` runs.

## 1. Confirm raw files are present

In [1]:
import os

DATA_PATH = "../data/raw/cicids2017"

csv_files = sorted([f for f in os.listdir(DATA_PATH) if f.endswith(".csv")])
print(f"Found {len(csv_files)} CSV files:\n")
for f in csv_files:
    size_mb = os.path.getsize(os.path.join(DATA_PATH, f)) / 1e6
    print(f"  {f:<60}  {size_mb:.1f} MB")

Found 8 CSV files:

  Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv              77.1 MB
  Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv          76.9 MB
  Friday-WorkingHours-Morning.pcap_ISCX.csv                     58.3 MB
  Monday-WorkingHours.pcap_ISCX.csv                             176.9 MB
  Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv   83.1 MB
  Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv        52.0 MB
  Tuesday-WorkingHours.pcap_ISCX.csv                            135.1 MB
  Wednesday-workingHours.pcap_ISCX.csv                          225.2 MB


## 2. Load one file and inspect the schema

We load a single CSV to check column names and types. CICIDS2017 CSVs have a known issue: column names contain leading/trailing whitespace, which must be stripped before any lookups.

In [2]:
import pandas as pd

sample_path = os.path.join(DATA_PATH, csv_files[0])
print(f"Loading: {csv_files[0]}")

df = pd.read_csv(sample_path, nrows=5000, low_memory=False)

print(f"\nShape (5,000-row sample): {df.shape}")
print(f"Columns before strip : {list(df.columns[:5])} ...")

# Strip whitespace from column names — this is required for CICIDS2017
df.columns = df.columns.str.strip()
print(f"Columns after strip  : {list(df.columns[:5])} ...")

Loading: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv

Shape (5,000-row sample): (5000, 79)
Columns before strip : [' Destination Port', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', 'Total Length of Fwd Packets'] ...
Columns after strip  : ['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets'] ...


In [3]:
# Full column list and dtypes
print(f"Total columns: {len(df.columns)}\n")
print(df.dtypes.to_string())

Total columns: 79

Destination Port                 int64
Flow Duration                    int64
Total Fwd Packets                int64
Total Backward Packets           int64
Total Length of Fwd Packets      int64
Total Length of Bwd Packets      int64
Fwd Packet Length Max            int64
Fwd Packet Length Min            int64
Fwd Packet Length Mean         float64
Fwd Packet Length Std          float64
Bwd Packet Length Max            int64
Bwd Packet Length Min            int64
Bwd Packet Length Mean         float64
Bwd Packet Length Std          float64
Flow Bytes/s                   float64
Flow Packets/s                 float64
Flow IAT Mean                  float64
Flow IAT Std                   float64
Flow IAT Max                     int64
Flow IAT Min                     int64
Fwd IAT Total                    int64
Fwd IAT Mean                   float64
Fwd IAT Std                    float64
Fwd IAT Max                      int64
Fwd IAT Min                      int64
Bwd IA

In [4]:
# First few rows
df.head(3)

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,54865,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,55054,109,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,55055,52,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


## 3. Label column — class distribution in one file

Each file captures one day of traffic. Monday is all-benign; attacks appear Tuesday–Friday.

In [5]:
print(f"Label distribution in: {csv_files[0]}\n")
label_counts = df["Label"].value_counts()
label_pct    = df["Label"].value_counts(normalize=True).mul(100).round(2)

label_summary = pd.DataFrame({"Count": label_counts, "Pct %": label_pct})
print(label_summary.to_string())

Label distribution in: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv

        Count  Pct %
Label               
BENIGN   5000  100.0


## 4. Data quality checks

Three known issues in raw CICIDS2017:
1. **Infinite values** — `Flow Bytes/s` and `Flow Packets/s` can be `inf` due to zero-duration flows
2. **Missing values (NaN)** — result from dropped packets or truncated flows
3. **Encoding artefacts** — the `Web Attack` label variants use a garbled unicode character instead of `–` (en-dash)

In [6]:
import numpy as np

numeric_cols = df.select_dtypes(include=[np.number])

inf_count = np.isinf(numeric_cols).sum().sum()
nan_count = df.isnull().sum().sum()

print(f"Rows with inf values : {np.isinf(numeric_cols).any(axis=1).sum():,}")
print(f"Total inf cells      : {inf_count:,}")
print(f"Total NaN cells      : {nan_count:,}")

print("\nColumns with inf values:")
inf_cols = numeric_cols.columns[np.isinf(numeric_cols).any()].tolist()
print(inf_cols if inf_cols else "  None")

print("\nTop columns with NaN:")
nan_by_col = df.isnull().sum().sort_values(ascending=False)
print(nan_by_col[nan_by_col > 0].head(10) if nan_by_col.any() else "  None")

Rows with inf values : 4
Total inf cells      : 8
Total NaN cells      : 0

Columns with inf values:
['Flow Bytes/s', 'Flow Packets/s']

Top columns with NaN:
  None


In [7]:
# Encoding artefacts: Web Attack labels use garbled unicode
web_attack_labels = [l for l in df["Label"].unique() if "Web Attack" in str(l) or "WebAttack" in str(l)]
print("Web Attack label variants found:")
for l in web_attack_labels:
    print(f"  repr: {repr(l)}")

print("\n→ ingest.py normalises these to WebAttack_BruteForce, WebAttack_XSS, WebAttack_SQLInjection")

Web Attack label variants found:

→ ingest.py normalises these to WebAttack_BruteForce, WebAttack_XSS, WebAttack_SQLInjection


## 5. Combined label distribution across all 8 files

After `ingest.py` runs and saves `cicids2017.parquet`, we can see the full dataset at once.

In [8]:
PROCESSED = "../data/processed/cicids2017.parquet"

if not os.path.exists(PROCESSED):
    print("cicids2017.parquet not found — run ingest.py first.")
    print("  python src/ingest.py")
else:
    full_df = pd.read_parquet(PROCESSED)
    print(f"Full dataset shape: {full_df.shape}")
    print(f"\nLabel distribution (all files combined):")
    counts = full_df["Label"].value_counts()
    pct    = full_df["Label"].value_counts(normalize=True).mul(100).round(2)
    print(pd.DataFrame({"Count": counts, "Pct %": pct}).to_string())

Full dataset shape: (2827876, 81)

Label distribution (all files combined):
                              Count  Pct %
Label                                     
BENIGN                      2271320  80.32
DoS Hulk                     230124   8.14
PortScan                     158804   5.62
DDoS                         128025   4.53
DoS GoldenEye                 10293   0.36
FTP-Patator                    7935   0.28
SSH-Patator                    5897   0.21
DoS slowloris                  5796   0.20
DoS Slowhttptest               5499   0.19
Bot                            1956   0.07
Web Attack � Brute Force       1507   0.05
Web Attack � XSS                652   0.02
Infiltration                     36   0.00
Web Attack � Sql Injection       21   0.00
Heartbleed                       11   0.00


In [9]:
# Attack types only (no BENIGN)
if os.path.exists(PROCESSED):
    attack_labels = sorted([l for l in full_df["Label"].unique() if l != "BENIGN"])
    print(f"{len(attack_labels)} distinct attack types:")
    for l in attack_labels:
        print(f"  {l}")

14 distinct attack types:
  Bot
  DDoS
  DoS GoldenEye
  DoS Hulk
  DoS Slowhttptest
  DoS slowloris
  FTP-Patator
  Heartbleed
  Infiltration
  PortScan
  SSH-Patator
  Web Attack � Brute Force
  Web Attack � Sql Injection
  Web Attack � XSS


## 6. Per-file breakdown

CICIDS2017 uses a temporal structure: one file per day/session. This drives our train/valid/test split strategy in `preprocess.py`.

In [10]:
if os.path.exists(PROCESSED):
    per_file = full_df.groupby("source_file")["Label"].value_counts().unstack(fill_value=0)
    per_file["TOTAL"] = per_file.sum(axis=1)
    per_file = per_file.sort_values("TOTAL", ascending=False)
    print(per_file[["TOTAL", "BENIGN"] + [c for c in per_file.columns if c not in ["TOTAL", "BENIGN"]]].to_string())

Label                                                         TOTAL  BENIGN   Bot    DDoS  DoS GoldenEye  DoS Hulk  DoS Slowhttptest  DoS slowloris  FTP-Patator  Heartbleed  Infiltration  PortScan  SSH-Patator  Web Attack � Brute Force  Web Attack � Sql Injection  Web Attack � XSS
source_file                                                                                                                                                                                                                                                                              
Wednesday-workingHours.pcap_ISCX.csv                         691406  439683     0       0          10293    230124              5499           5796            0          11             0         0            0                         0                           0                 0
Monday-WorkingHours.pcap_ISCX.csv                            529481  529481     0       0              0         0                 0              0       

## 7. Feature statistics — top features by variance

High-variance features carry the most signal for our models. This cell ranks them.

In [11]:
if os.path.exists(PROCESSED):
    numeric = full_df.select_dtypes(include=[np.number])
    top_var = numeric.var().nlargest(15)
    
    print("Top 15 features by variance:")
    print(top_var.to_string())
    
    print("\nDescriptive stats for top 10 features:")
    print(numeric[top_var.head(10).index].describe().T[["mean","std","min","max"]].round(3).to_string())

Top 15 features by variance:
Flow Duration          1.133501e+15
Fwd IAT Total          1.128265e+15
Bwd IAT Total          8.265309e+14
Flow Bytes/s           6.728917e+14
Fwd IAT Max            6.022066e+14
Flow IAT Max           5.987900e+14
Idle Max               5.942705e+14
Idle Mean              5.588765e+14
Idle Min               5.463391e+14
Fwd Header Length      4.436722e+14
Fwd Header Length.1    4.436722e+14
Bwd IAT Max            2.947745e+14
Fwd IAT Std            9.299475e+13
Fwd IAT Mean           9.082447e+13
Bwd IAT Mean           7.905904e+13

Descriptive stats for top 10 features:
                           mean           std           min           max
Flow Duration      1.480065e+07  3.366750e+07 -1.300000e+01  1.200000e+08
Fwd IAT Total      1.449765e+07  3.358966e+07  0.000000e+00  1.200000e+08
Bwd IAT Total      9.903861e+06  2.874945e+07  0.000000e+00  1.200000e+08
Flow Bytes/s       1.491719e+06  2.594016e+07 -2.610000e+08  2.071000e+09
Fwd IAT Max        9.

## Summary

| Finding | Detail |
|---|---|
| Dataset | CICIDS2017 — 8 CSV files, 5 days of traffic |
| Total flows | ~2.83 million (after cleaning) |
| Features | 78 numeric + 1 label |
| Label column | `Label` (with leading whitespace in raw files) |
| Attack types | 14 distinct classes + BENIGN |
| Imbalance | ~80% BENIGN, top attack is DoS Hulk (~8%) |
| Key issues | Whitespace in column names, inf values in `Flow Bytes/s`, garbled Web Attack labels |
| Fixed by | `ingest.py` strips columns, drops inf/NaN; `preprocess.py` normalises labels |

**Next:** `02_cicids_dataset_summary.ipynb` — temporal split inspection and feature analysis.